# Tutorial 4: Working with JANIS (JAva-based Nuclear Data Information System)

## Overview

JANIS is a web-based system maintained by the OECD NEA that provides unified access to multiple nuclear data libraries. Instead of downloading files from different sources, you can query multiple databases through one interface.

### What you'll learn:
- How to download data from multiple libraries using JANIS
- How to compare REAL data from different sources (ENDF, JEFF, JENDL, TENDL)
- How to quantify inter-library differences and uncertainties
- How to create ML-ready datasets from multi-library comparisons
- Understanding systematic differences between evaluation methodologies

### Prerequisites:
```bash
pip install matplotlib numpy pandas scipy
```

### ⚠️ IMPORTANT: This tutorial requires REAL data from multiple libraries!
You must download actual evaluations from at least 3 different libraries using JANIS.

## 1. Understanding JANIS

### Key Features:

- **Multi-library Access**: Query ENDF, JEFF, JENDL, TENDL, and more
- **Web Interface**: https://www.oecd-nea.org/janisweb/
- **Interactive Plots**: View cross-sections in browser
- **Data Export**: Download data in CSV format
- **Comparison Tools**: Side-by-side library comparison

### Available Libraries in JANIS:

- **ENDF/B-VIII.0** (USA) - Most widely used
- **JEFF-3.3** (Europe) - Used by European reactors
- **JENDL-5.0** (Japan) - Japanese evaluation
- **TENDL-2021** (PSI/IAEA) - Automated TALYS evaluations
- **CENDL-3.2** (China) - Chinese evaluation
- **ROSFOND-2010** (Russia) - Russian evaluation
- **EXFOR** (Experimental) - Experimental measurements

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from scipy.interpolate import interp1d

# Create data directory
data_dir = Path('../data/janis')
data_dir.mkdir(parents=True, exist_ok=True)

print("Setup complete!")
print(f"Data directory: {data_dir.absolute()}")

## 2. 📥 DOWNLOADING REAL MULTI-LIBRARY DATA

### Step-by-Step Instructions:

#### Download U-235 Fission from Multiple Libraries:

1. **Visit JANIS:**
   - Go to: https://www.oecd-nea.org/janisweb/

2. **For EACH library (repeat 4 times):**

   **Library 1: ENDF/B-VIII.0**
   - Click "Search" → "Cross Sections"
   - Select isotope: U-235
   - Select reaction: (n,f) for fission
   - **Check ONLY "ENDF/B-VIII.0"** library
   - Click "Plot Data"
   - Click "Export" → "CSV"
   - Save as: `u235_fission_endf.csv`

   **Library 2: JEFF-3.3**
   - Same steps, but check **ONLY "JEFF-3.3"**
   - Save as: `u235_fission_jeff.csv`

   **Library 3: JENDL-5.0**
   - Same steps, but check **ONLY "JENDL-5.0"**
   - Save as: `u235_fission_jendl.csv`

   **Library 4: TENDL-2021**
   - Same steps, but check **ONLY "TENDL-2021"**
   - Save as: `u235_fission_tendl.csv`

   **Optional: CENDL-3.2**
   - Same steps, but check **ONLY "CENDL-3.2"**
   - Save as: `u235_fission_cendl.csv`

3. **Save all files to:** `../data/janis/`

### 📋 Required Files (minimum 3 libraries):
- `u235_fission_endf.csv` - ENDF/B-VIII.0 (USA)
- `u235_fission_jeff.csv` - JEFF-3.3 (Europe)
- `u235_fission_jendl.csv` - JENDL-5.0 (Japan)
- `u235_fission_tendl.csv` - TENDL-2021 (TALYS) - Optional but recommended
- `u235_fission_cendl.csv` - CENDL-3.2 (China) - Optional

### ⚠️ Important:
- Download each library SEPARATELY (one at a time)
- Use consistent naming scheme
- All files should have the same reaction (n,f) and isotope (U-235)

## 3. Loading Real Multi-Library Data

Let's load data from all available libraries you downloaded.

In [ ]:
# Define library files
library_files = {
    'ENDF/B-VIII.0': data_dir / 'u235_fission_endf.csv',
    'JEFF-3.3': data_dir / 'u235_fission_jeff.csv',
    'JENDL-5.0': data_dir / 'u235_fission_jendl.csv',
    'TENDL-2021': data_dir / 'u235_fission_tendl.csv',
    'CENDL-3.2': data_dir / 'u235_fission_cendl.csv'
}

# Check which files exist
available_libraries = {}
missing_libraries = []

for lib_name, file_path in library_files.items():
    if file_path.exists():
        available_libraries[lib_name] = file_path
    else:
        missing_libraries.append((lib_name, file_path.name))

if len(available_libraries) < 3:
    print("❌ ERROR: Insufficient library data!")
    print(f"\nFound {len(available_libraries)} libraries, need at least 3 for comparison.")
    print(f"\nAvailable: {', '.join(available_libraries.keys())}")
    print(f"\nMissing:")
    for lib, filename in missing_libraries:
        print(f"  - {lib}: {filename}")
    print(f"\nExpected directory: {data_dir.absolute()}")
    print("\n📥 PLEASE DOWNLOAD REAL DATA FROM MULTIPLE LIBRARIES:")
    print("\n   Using JANIS:")
    print("   1. Go to https://www.oecd-nea.org/janisweb/")
    print("   2. For EACH library:")
    print("      a. Search → Cross Sections")
    print("      b. Select U-235, reaction (n,f)")
    print("      c. Check ONLY ONE library at a time")
    print("      d. Plot Data → Export → CSV")
    print("      e. Save with library-specific name")
    print("   3. Repeat for at least 3 different libraries")
    print(f"   4. Save all to: {data_dir.absolute()}")
    print("\n⚠️ This tutorial REQUIRES data from at least 3 libraries!")
    raise FileNotFoundError(f"Need at least 3 libraries, found {len(available_libraries)}")

print(f"✓ Found {len(available_libraries)} libraries:")
for lib in available_libraries.keys():
    print(f"  - {lib}")

if missing_libraries:
    print(f"\nOptional libraries not found: {', '.join([lib for lib, _ in missing_libraries])}")

## 4. Parse and Load All Library Data

Load and standardize data from all available libraries.

In [ ]:
def load_library_csv(file_path, library_name):
    """
    Load library data from JANIS CSV export.
    JANIS CSV format may vary slightly, so we identify columns intelligently.
    """
    try:
        df = pd.read_csv(file_path, comment='#', skipinitialspace=True)
        
        # Identify energy and cross-section columns
        energy_cols = [col for col in df.columns if 'energ' in col.lower() or col.lower() == 'e']
        xs_cols = [col for col in df.columns if any(x in col.lower() for x in ['cross', 'xs', 'sigma', 'data'])]
        
        if not energy_cols or not xs_cols:
            # Fallback: assume first two columns
            print(f"  ⚠️ Using default columns for {library_name}")
            energy_col = df.columns[0]
            xs_col = df.columns[1]
        else:
            energy_col = energy_cols[0]
            xs_col = xs_cols[0]
        
        # Create standardized dataframe
        result = pd.DataFrame()
        result['Energy_eV'] = pd.to_numeric(df[energy_col], errors='coerce')
        result['CrossSection_barns'] = pd.to_numeric(df[xs_col], errors='coerce')
        
        # Remove NaN and sort
        result = result.dropna()
        result = result.sort_values('Energy_eV').reset_index(drop=True)
        
        print(f"  ✓ {library_name}: {len(result)} points")
        print(f"    Energy: {result['Energy_eV'].min():.2e} - {result['Energy_eV'].max():.2e} eV")
        print(f"    XS: {result['CrossSection_barns'].min():.3f} - {result['CrossSection_barns'].max():.3f} barns")
        
        return result
        
    except Exception as e:
        print(f"  ❌ Error loading {library_name}: {e}")
        raise

# Load all available libraries
library_data = {}

print("Loading library data...\n")
for lib_name, file_path in available_libraries.items():
    df = load_library_csv(file_path, lib_name)
    library_data[lib_name] = df
    print()

print("="*60)
print(f"REAL DATA FROM {len(library_data)} LIBRARIES LOADED SUCCESSFULLY")
print("="*60)

## 5. Multi-Library Comparison Plot [REAL DATA]

Let's visualize all libraries on the same plot.

In [ ]:
plt.figure(figsize=(14, 8))

# Define colors and line styles for different libraries
colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown', 'pink']
linestyles = ['-', '--', '-.', ':', '-', '--', '-.']

for i, (lib_name, df) in enumerate(library_data.items()):
    plt.loglog(df['Energy_eV'], df['CrossSection_barns'], 
               label=lib_name, 
               color=colors[i % len(colors)], 
               linestyle=linestyles[i % len(linestyles)],
               linewidth=2.5,
               alpha=0.8)

plt.xlabel('Neutron Energy (eV)', fontsize=14, fontweight='bold')
plt.ylabel('Fission Cross-section (barns)', fontsize=14, fontweight='bold')
plt.title(f'U-235 Fission Cross-Section: {len(library_data)}-Library Comparison - REAL DATA', 
          fontsize=16, fontweight='bold', pad=20)
plt.legend(fontsize=11, loc='best', framealpha=0.9)
plt.grid(True, alpha=0.3, which='both')

# Add watermark
plt.text(0.98, 0.02, '[REAL DATA]', 
         transform=plt.gca().transAxes,
         fontsize=12, color='red', fontweight='bold',
         ha='right', va='bottom',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(data_dir / 'janis_multi_library_comparison_REAL.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Multi-library comparison saved: {data_dir / 'janis_multi_library_comparison_REAL.png'}")

## 6. Ratio Analysis: Compare All Libraries to Reference [REAL DATA]

Let's use ENDF/B-VIII.0 as the reference and calculate ratios for all other libraries.

In [ ]:
# Use ENDF as reference (if available), otherwise use first library
if 'ENDF/B-VIII.0' in library_data:
    reference_name = 'ENDF/B-VIII.0'
else:
    reference_name = list(library_data.keys())[0]
    print(f"⚠️ ENDF not available, using {reference_name} as reference")

reference_data = library_data[reference_name]

# Create comparison plot with ratios
fig, axes = plt.subplots(2, 1, figsize=(14, 10), 
                         gridspec_kw={'height_ratios': [3, 2]})

# Top panel: Cross-sections
for i, (lib_name, df) in enumerate(library_data.items()):
    axes[0].loglog(df['Energy_eV'], df['CrossSection_barns'], 
                   label=lib_name, 
                   color=colors[i % len(colors)], 
                   linestyle=linestyles[i % len(linestyles)],
                   linewidth=2.5,
                   alpha=0.8)

axes[0].set_ylabel('Fission Cross-section (barns)', fontsize=14, fontweight='bold')
axes[0].set_title(f'U-235 Fission: All Libraries with Ratios to {reference_name} - REAL DATA', 
                  fontsize=16, fontweight='bold', pad=20)
axes[0].legend(fontsize=10, loc='best')
axes[0].grid(True, alpha=0.3, which='both')

# Bottom panel: Ratios to reference
for i, (lib_name, df) in enumerate(library_data.items()):
    if lib_name == reference_name:
        continue  # Skip reference itself
    
    # Interpolate to reference energy grid
    f_interp = interp1d(df['Energy_eV'], df['CrossSection_barns'], 
                        kind='linear', bounds_error=False, fill_value=np.nan)
    xs_interp = f_interp(reference_data['Energy_eV'])
    ratio = xs_interp / reference_data['CrossSection_barns'].values
    
    axes[1].semilogx(reference_data['Energy_eV'], ratio, 
                     label=f'{lib_name}/{reference_name}', 
                     color=colors[i % len(colors)], 
                     linewidth=2)

axes[1].axhline(1.0, color='black', linestyle='-', linewidth=1.5, alpha=0.7, label='Perfect agreement')
axes[1].axhline(1.02, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='±2%')
axes[1].axhline(0.98, color='gray', linestyle='--', linewidth=1, alpha=0.5)
axes[1].set_xlabel('Neutron Energy (eV)', fontsize=14, fontweight='bold')
axes[1].set_ylabel(f'Ratio to {reference_name}', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10, loc='best')
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0.97, 1.03)

plt.tight_layout()
plt.savefig(data_dir / 'janis_ratio_analysis_REAL.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Ratio analysis saved: {data_dir / 'janis_ratio_analysis_REAL.png'}")

## 7. Statistical Comparison [REAL DATA]

Calculate comprehensive statistics for each library relative to the reference.

In [ ]:
# Calculate statistics for each library
stats_data = []

for lib_name, df in library_data.items():
    if lib_name == reference_name:
        # Reference library
        stats_data.append({
            'Library': lib_name,
            'Data_Points': len(df),
            'Mean_Ratio': 1.0,
            'Std_Ratio': 0.0,
            'Max_Deviation_%': 0.0,
            'Status': 'Reference'
        })
    else:
        # Interpolate to reference grid
        f_interp = interp1d(df['Energy_eV'], df['CrossSection_barns'], 
                            kind='linear', bounds_error=False, fill_value=np.nan)
        xs_interp = f_interp(reference_data['Energy_eV'])
        ratio = xs_interp / reference_data['CrossSection_barns'].values
        
        # Remove NaN values
        valid_ratio = ratio[~np.isnan(ratio)]
        
        stats_data.append({
            'Library': lib_name,
            'Data_Points': len(df),
            'Mean_Ratio': np.mean(valid_ratio),
            'Std_Ratio': np.std(valid_ratio),
            'Max_Deviation_%': np.max(np.abs(valid_ratio - 1.0)) * 100,
            'Status': 'Comparison'
        })

df_stats = pd.DataFrame(stats_data)

print("="*80)
print(f"STATISTICAL COMPARISON (relative to {reference_name}) [REAL DATA]")
print("="*80)
print(df_stats.to_string(index=False))

# Save statistics
csv_file = data_dir / 'janis_library_statistics_REAL.csv'
df_stats.to_csv(csv_file, index=False)
print(f"\n✓ Statistics saved: {csv_file}")

## 8. Energy Region Analysis [REAL DATA]

Examine how libraries differ in specific energy regions.

In [ ]:
# Define energy regions
regions = {
    'Thermal': (0.01, 1.0),
    'Epithermal': (1.0, 100.0),
    'Resonance': (100.0, 10000.0),
    'Fast': (10000.0, 1e7)
}

# Calculate mean ratios for each region
region_stats = []

for lib_name, df in library_data.items():
    if lib_name == reference_name:
        continue
    
    # Interpolate to reference grid
    f_interp = interp1d(df['Energy_eV'], df['CrossSection_barns'], 
                        kind='linear', bounds_error=False, fill_value=np.nan)
    xs_interp = f_interp(reference_data['Energy_eV'])
    ratio = xs_interp / reference_data['CrossSection_barns'].values
    
    for region_name, (e_min, e_max) in regions.items():
        # Find points in this energy range
        mask = (reference_data['Energy_eV'] >= e_min) & (reference_data['Energy_eV'] <= e_max)
        
        if np.any(mask):
            region_ratio = ratio[mask]
            valid_ratio = region_ratio[~np.isnan(region_ratio)]
            
            if len(valid_ratio) > 0:
                region_stats.append({
                    'Library': lib_name,
                    'Region': region_name,
                    'Mean_Ratio': np.mean(valid_ratio),
                    'Std_Ratio': np.std(valid_ratio)
                })

df_regions = pd.DataFrame(region_stats)

# Pivot for easier visualization
df_pivot = df_regions.pivot(index='Library', columns='Region', values='Mean_Ratio')

print("\n" + "="*80)
print("MEAN RATIO BY ENERGY REGION [REAL DATA]")
print("="*80)
print(df_pivot)

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
df_pivot.plot(kind='bar', ax=ax, width=0.8, color=['blue', 'orange', 'green', 'red'])
ax.axhline(1.0, color='black', linestyle='--', linewidth=2, label='Perfect agreement')
ax.set_ylabel(f'Mean Ratio to {reference_name}', fontsize=12, fontweight='bold')
ax.set_xlabel('Library', fontsize=12, fontweight='bold')
ax.set_title('Library Comparison by Energy Region - REAL DATA', fontsize=14, fontweight='bold')
ax.legend(title='Energy Region', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(data_dir / 'janis_region_comparison_REAL.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Region analysis saved: {data_dir / 'janis_region_comparison_REAL.png'}")

## 9. Uncertainty Quantification from Library Spread [REAL DATA]

The spread between different libraries provides a measure of evaluation uncertainty.

In [ ]:
# Interpolate all libraries to a common energy grid
# Find common energy range
e_min = max([df['Energy_eV'].min() for df in library_data.values()])
e_max = min([df['Energy_eV'].max() for df in library_data.values()])
energy_common = np.logspace(np.log10(e_min), np.log10(e_max), 1000)

# Interpolate all libraries to common grid
xs_all = []
for lib_name, df in library_data.items():
    f_interp = interp1d(df['Energy_eV'], df['CrossSection_barns'], 
                        kind='linear', fill_value='extrapolate')
    xs_all.append(f_interp(energy_common))

xs_all = np.array(xs_all)

# Calculate statistics across libraries
xs_mean = np.mean(xs_all, axis=0)
xs_std = np.std(xs_all, axis=0)
xs_min = np.min(xs_all, axis=0)
xs_max = np.max(xs_all, axis=0)
xs_cov = (xs_std / xs_mean) * 100  # Coefficient of variation in %

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Top: Mean ± std
ax1.loglog(energy_common, xs_mean, 'k-', linewidth=2.5, label='Mean across libraries', zorder=3)
ax1.fill_between(energy_common, 
                 xs_mean - xs_std,
                 xs_mean + xs_std,
                 alpha=0.3, color='blue', label='±1σ (library spread)', zorder=2)
ax1.fill_between(energy_common,
                 xs_min,
                 xs_max,
                 alpha=0.15, color='red', label='Min-Max range', zorder=1)

ax1.set_ylabel('Fission Cross-section (barns)', fontsize=14, fontweight='bold')
ax1.set_title(f'U-235 Fission: Evaluation Uncertainty from {len(library_data)}-Library Spread - REAL DATA', 
              fontsize=16, fontweight='bold', pad=20)
ax1.legend(fontsize=12, loc='best')
ax1.grid(True, alpha=0.3, which='both')

# Bottom: Coefficient of variation
ax2.semilogx(energy_common, xs_cov, 'purple', linewidth=2.5)
ax2.set_xlabel('Neutron Energy (eV)', fontsize=14, fontweight='bold')
ax2.set_ylabel('Coefficient of Variation (%)', fontsize=12, fontweight='bold')
ax2.set_title('Relative Spread Between Libraries', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(data_dir / 'janis_uncertainty_analysis_REAL.png', dpi=150, bbox_inches='tight')
plt.show()

print("="*80)
print("UNCERTAINTY ANALYSIS [REAL DATA]")
print("="*80)
print(f"Mean coefficient of variation: {np.mean(xs_cov):.3f}%")
print(f"Max coefficient of variation: {np.max(xs_cov):.3f}%")
print(f"Energy with max disagreement: {energy_common[np.argmax(xs_cov)]:.2e} eV")
print(f"\n✓ Uncertainty analysis saved: {data_dir / 'janis_uncertainty_analysis_REAL.png'}")

## 10. Creating ML Dataset [REAL DATA]

Create a comprehensive ML-ready dataset with all libraries.

In [ ]:
# Create combined dataframe
df_ml = pd.DataFrame({'Energy_eV': energy_common})

# Add each library's cross-section
for lib_name, df in library_data.items():
    # Sanitize library name for column
    col_name = lib_name.replace('/', '_').replace('-', '_').replace('.', '_')
    
    f_interp = interp1d(df['Energy_eV'], df['CrossSection_barns'], 
                        kind='linear', fill_value='extrapolate')
    df_ml[col_name] = f_interp(energy_common)

# Add statistical features
lib_columns = [col for col in df_ml.columns if col != 'Energy_eV']
df_ml['Mean_XS'] = df_ml[lib_columns].mean(axis=1)
df_ml['Std_XS'] = df_ml[lib_columns].std(axis=1)
df_ml['Min_XS'] = df_ml[lib_columns].min(axis=1)
df_ml['Max_XS'] = df_ml[lib_columns].max(axis=1)
df_ml['Range_XS'] = df_ml['Max_XS'] - df_ml['Min_XS']
df_ml['CoV_%'] = (df_ml['Std_XS'] / df_ml['Mean_XS']) * 100
df_ml['Log10_Energy'] = np.log10(df_ml['Energy_eV'])

# Energy region classification
def classify_energy_region(energy):
    if energy < 1:
        return 'Thermal'
    elif energy < 100:
        return 'Epithermal'
    elif energy < 10000:
        return 'Resonance'
    else:
        return 'Fast'

df_ml['Energy_Region'] = df_ml['Energy_eV'].apply(classify_energy_region)

# Save ML dataset
ml_file = data_dir / 'janis_combined_libraries_ML_REAL.csv'
df_ml.to_csv(ml_file, index=False)

print("="*80)
print("ML DATASET PREPARED [REAL DATA]")
print("="*80)
print(f"\nTotal samples: {len(df_ml)}")
print(f"\nLibraries included: {', '.join(library_data.keys())}")
print(f"\nFeatures:")
print(f"  - Cross-sections from {len(library_data)} libraries")
print(f"  - Statistical features: mean, std, min, max, range, CoV")
print(f"  - Energy (eV and log10)")
print(f"  - Energy region classification")
print(f"\nData points by energy region:")
print(df_ml['Energy_Region'].value_counts())
print(f"\nCoefficient of Variation by region:")
print(df_ml.groupby('Energy_Region')['CoV_%'].describe())
print(f"\n✓ ML dataset saved: {ml_file}")

## 11. Key Takeaways

### What We Learned:

1. **JANIS provides unified access** to multiple nuclear data libraries
2. **Real multi-library comparison** reveals evaluation uncertainties and systematic differences
3. **Library spread quantifies uncertainty** - Different evaluations provide different results
4. **Energy region dependence** - Libraries may agree in some regions but disagree in others
5. **No single "correct" library** - Different countries use different evaluations
6. **Systematic methodology differences** - TALYS (TENDL) vs manual evaluation (ENDF, JEFF, JENDL)

### Best Practices for Nuclear Data:

- **Always compare multiple libraries** for critical applications
- **Use library spread as uncertainty estimate** when no experimental data available
- **Investigate high-disagreement regions** - May indicate need for new measurements
- **Validate with EXFOR** - Compare evaluations with experimental data
- **Document which library used** - Critical for reproducibility
- **Understand regional preferences** - ENDF (US), JEFF (Europe), JENDL (Japan)

### Important for Machine Learning:

- **Ensemble methods**: Use multiple libraries for robust predictions
- **Uncertainty quantification**: Library spread provides realistic uncertainty bounds
- **Data augmentation**: Multi-library data increases training set diversity
- **Outlier detection**: Identify libraries/regions with anomalous behavior
- **Transfer learning**: Learn from systematic differences between evaluations

## Next Steps

- Download multi-library data for other isotopes (Pu-239, Fe-56, etc.)
- Compare with EXFOR experimental data (Tutorial 2)
- Train ML models using multi-library ensemble
- Investigate specific energy regions where libraries disagree
- Develop systematic bias correction methods
- Create automated validation workflows

## Resources

- **JANIS Web:** https://www.oecd-nea.org/janisweb/
- **OECD NEA:** https://www.oecd-nea.org/
- **ENDF Database:** https://www.nndc.bnl.gov/endf/
- **JEFF Database:** https://www.oecd-nea.org/dbdata/jeff/
- **JENDL Database:** https://wwwndc.jaea.go.jp/jendl/
- **Nuclear Data Services:** https://www-nds.iaea.org/